# Análisis Exploratorio de Datos - Consumo de Biodiesel
## Repsol Capstone Project - Semana 1

Análisis de patrones de consumo de biodiesel en España (2023-2025)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
DATA_INPUTS  = REPO_ROOT / 'data' / 'inputs'
FIGS         = REPO_ROOT / 'reports' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

# Load cleaned biodiesel consumption: all CCAAs + ESPAÑA national total, 2023-2025
df = pd.read_csv(DATA_INPUTS / 'consumo_biodiesel_ccaa.csv')
df['Fecha'] = pd.to_datetime(df['Fecha'])
df = df.sort_values('Fecha').reset_index(drop=True)

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['Fecha'].min().strftime('%Y-%m')} \u2192 {df['Fecha'].max().strftime('%Y-%m')}")
print(f"CCAAs ({df['CCAA'].nunique()}): {sorted(df['CCAA'].unique())}")


## 1. Setup — Carga de Datos

Datos cargados desde `data/inputs/consumo_biodiesel_ccaa.csv` (720 filas, 3 columnas: Fecha, CCAA, Consumo_Tm).

In [ ]:
print("=" * 60)
print("EXPLORACIÓN DE DATOS - BIODIESEL CONSUMO")
print("=" * 60)

print(f"\nDIMENSIONES:")
print(f"   Filas: {len(df)}")
print(f"   Período: {df['Fecha'].min().strftime('%Y-%m')} a {df['Fecha'].max().strftime('%Y-%m')}")
print(f"   CCAA/Nacional: {df['CCAA'].nunique()}")

print(f"\nESTADÍSTICAS DE CONSUMO (Tm):")
print(df['Consumo_Tm'].describe())

print(f"\nVALORES FALTANTES:")
print(df.isnull().sum())

print(f"\nPRIMERAS 10 FILAS:")
print(df.head(10))


## 2. Consumo Nacional

In [ ]:
# 1. CONSUMO NACIONAL A LO LARGO DEL TIEMPO
df_nacional = df[df['CCAA'] == 'ESPAÑA'].copy()

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Gráfico 1: Línea temporal nacional
axes[0].plot(df_nacional['Fecha'], df_nacional['Consumo_Tm'], 
             linewidth=2, marker='o', color='#FF6B35', markersize=4)
axes[0].set_title('Consumo de Biodiesel en España (2023-2025)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('Consumo (Tm)')
axes[0].grid(True, alpha=0.3)
axes[0].fill_between(df_nacional['Fecha'], df_nacional['Consumo_Tm'], alpha=0.2, color='#FF6B35')

# Gráfico 2: Distribución mensual
axes[1].bar(df_nacional['Fecha'], df_nacional['Consumo_Tm'], 
            color='#004E89', width=20, alpha=0.7)
axes[1].set_title('Consumo Mensual (Barras)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Consumo (Tm)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGS / '01_consumo_nacional.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: 01_consumo_nacional.png")

## 3. Análisis Regional — Top 5 CCAA

In [ ]:
# 2. CONSUMO POR COMUNIDAD AUTÓNOMA (TOP 5)
df_regional = df[df['CCAA'] != 'ESPAÑA'].copy()

# Calcular consumo total por CCAA
ccaa_consumo = df_regional.groupby('CCAA')['Consumo_Tm'].sum().sort_values(ascending=False)

print("🏆 TOP 10 CCAA por consumo total (2023-2025):")
print(ccaa_consumo.head(10))

# Visualizar TOP 5
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Top 5 CCAA
top5_ccaa = ccaa_consumo.head(5)
axes[0].barh(top5_ccaa.index, top5_ccaa.values, color='#1f77b4')
axes[0].set_title('Top 5 CCAA - Consumo Total Biodiesel (2023-2025)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Consumo Total (Tm)')
axes[0].grid(True, alpha=0.3, axis='x')

# Gráfico 2: Línea temporal Top 5
for ccaa in top5_ccaa.index:
    df_ccaa = df[df['CCAA'] == ccaa].sort_values('Fecha')
    axes[1].plot(df_ccaa['Fecha'], df_ccaa['Consumo_Tm'], 
                 marker='o', label=ccaa, linewidth=2, markersize=3)

axes[1].set_title('Evolución Temporal - Top 5 CCAA', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Consumo (Tm)')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS / '02_consumo_regional.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico guardado: 02_consumo_regional.png")

## 4. Estacionalidad y Tendencia

In [ ]:
# 3. ANÁLISIS DE ESTACIONALIDAD Y TENDENCIA
df_nacional = df[df['CCAA'] == 'ESPAÑA'].copy()

# Extraer mes y año
df_nacional['Mes'] = df_nacional['Fecha'].dt.month
df_nacional['Año'] = df_nacional['Fecha'].dt.year
df_nacional['Mes_Nombre'] = df_nacional['Fecha'].dt.strftime('%B')

# Consumo promedio por mes (estacionalidad)
consumo_por_mes = df_nacional.groupby('Mes')['Consumo_Tm'].mean()
mes_nombres = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
               'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gráfico 1: Estacionalidad (promedio por mes)
axes[0, 0].bar(range(1, 13), consumo_por_mes.values, color='#2ecc71', alpha=0.7)
axes[0, 0].set_title('Estacionalidad - Consumo Promedio por Mes', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Mes')
axes[0, 0].set_ylabel('Consumo Promedio (Tm)')
axes[0, 0].set_xticks(range(1, 13))
axes[0, 0].set_xticklabels([m[:3] for m in mes_nombres], rotation=45)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Gráfico 2: Consumo por año
consumo_por_año = df_nacional.groupby('Año')['Consumo_Tm'].sum()
axes[0, 1].bar(consumo_por_año.index, consumo_por_año.values, color='#e74c3c', alpha=0.7, width=0.6)
axes[0, 1].set_title('Consumo Total por Año', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Año')
axes[0, 1].set_ylabel('Consumo Total (Tm)')
axes[0, 1].set_xticks(consumo_por_año.index)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Gráfico 3: Box plot por mes (variabilidad)
df_nacional_sorted = df_nacional.sort_values('Mes')
df_nacional_sorted['Mes_Nombre_Short'] = df_nacional_sorted['Mes'].map({i: m[:3] for i, m in enumerate(mes_nombres, 1)})

box_data = [df_nacional[df_nacional['Mes'] == i]['Consumo_Tm'].values for i in range(1, 13)]
bp = axes[1, 0].boxplot(box_data, labels=[m[:3] for m in mes_nombres], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#3498db')
    patch.set_alpha(0.7)
axes[1, 0].set_title('Variabilidad Mensual (Box Plot)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Mes')
axes[1, 0].set_ylabel('Consumo (Tm)')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Gráfico 4: Distribución general
axes[1, 1].hist(df_nacional['Consumo_Tm'], bins=20, color='#9b59b6', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Distribución del Consumo Total', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Consumo (Tm)')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].axvline(df_nacional['Consumo_Tm'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_nacional["Consumo_Tm"].mean():.0f} Tm')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGS / '03_estacionalidad_tendencia.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: 03_estacionalidad_tendencia.png")

# Estadísticas por mes
print("\n📊 CONSUMO PROMEDIO POR MES:")
for mes, valor in consumo_por_mes.items():
    print(f"   {mes_nombres[mes-1]}: {valor:.0f} Tm")

print("\n📊 CONSUMO TOTAL POR AÑO:")
for año, valor in consumo_por_año.items():
    print(f"   {año}: {valor:.0f} Tm")

## 5. Patrones Temporales

In [ ]:
# 4. CORRELACIÓN Y RESUMEN ESTADÍSTICO FINAL
df_nacional = df[df['CCAA'] == 'ESPAÑA'].copy()

# Crear features temporales
df_nacional['Mes'] = df_nacional['Fecha'].dt.month
df_nacional['Trimestre'] = df_nacional['Fecha'].dt.quarter
df_nacional['Año'] = df_nacional['Fecha'].dt.year
df_nacional['Día_Año'] = df_nacional['Fecha'].dt.dayofyear

# Matriz de correlación
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Scatter - Mes vs Consumo
axes[0].scatter(df_nacional['Mes'], df_nacional['Consumo_Tm'], 
                s=100, alpha=0.6, c=df_nacional['Año'], cmap='viridis')
axes[0].set_title('Consumo por Mes (coloreado por Año)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('Consumo (Tm)')
axes[0].set_xticks(range(1, 13))
axes[0].grid(True, alpha=0.3)
cbar = plt.colorbar(axes[0].collections[0], ax=axes[0])
cbar.set_label('Año')

# Gráfico 2: Consumo por trimestre
consumo_trimestre = df_nacional.groupby('Trimestre')['Consumo_Tm'].mean()
colores_trim = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
axes[1].bar(consumo_trimestre.index, consumo_trimestre.values, 
            color=colores_trim, alpha=0.7, width=0.6)
axes[1].set_title('Consumo Promedio por Trimestre', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Trimestre')
axes[1].set_ylabel('Consumo Promedio (Tm)')
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4'])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGS / '04_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: 04_correlaciones.png")

# RESUMEN FINAL
print("\n" + "="*60)
print("RESUMEN ESTADÍSTICO FINAL - EDA COMPLETADO")
print("="*60)

print(f"\n📈 DATOS GENERALES:")
print(f"   Período: {df_nacional['Fecha'].min().strftime('%Y-%m-%d')} a {df_nacional['Fecha'].max().strftime('%Y-%m-%d')}")
print(f"   Total meses: {len(df_nacional)}")
print(f"   Consumo total España (3 años): {df_nacional['Consumo_Tm'].sum():,.0f} Tm")
print(f"   Consumo promedio mensual: {df_nacional['Consumo_Tm'].mean():.0f} Tm")
print(f"   Consumo mínimo: {df_nacional['Consumo_Tm'].min():.0f} Tm")
print(f"   Consumo máximo: {df_nacional['Consumo_Tm'].max():.0f} Tm")
print(f"   Desv. estándar: {df_nacional['Consumo_Tm'].std():.0f} Tm")

print(f"\n📊 ESTACIONALIDAD:")
mes_max = df_nacional.groupby('Mes')['Consumo_Tm'].mean().idxmax()
mes_min = df_nacional.groupby('Mes')['Consumo_Tm'].mean().idxmin()
mes_nombres = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio',
               'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
print(f"   Mes con más consumo: {mes_nombres[mes_max-1]}")
print(f"   Mes con menos consumo: {mes_nombres[mes_min-1]}")

print(f"\n📅 TENDENCIA ANUAL:")
for año in sorted(df_nacional['Año'].unique()):
    consumo_año = df_nacional[df_nacional['Año'] == año]['Consumo_Tm'].sum()
    print(f"   {año}: {consumo_año:,.0f} Tm")

print(f"\n🔍 TRIMESTRES:")
for trim in sorted(df_nacional['Trimestre'].unique()):
    consumo_trim = df_nacional[df_nacional['Trimestre'] == trim]['Consumo_Tm'].mean()
    print(f"   Q{trim}: {consumo_trim:.0f} Tm (promedio)")

print(f"\n✅ EDA COMPLETADO - Archivos guardados en /reports/figures/")
print("="*60)

## 6. Conclusiones

- Crecimiento explosivo: ×80 en 3 años (2023-2025)
- Concentración en 5 CCAA: >90% del consumo total
- Estacionalidad marcada: pico en Q4 (Oct-Dic)
- Madrid, Cataluña y Andalucía lideran la adopción